In [1]:
import pandas as pd,numpy as np,json
import getpass,os
from langchain.chat_models import init_chat_model
from ai_patterns_mining import parse_json_safe

/home/hasinthaka/Documents/Projects/AI/Pattern Mining/pipeline/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_csv('./clustered_dataset/umap_clustered_dataset-all v2.0.csv')

In [3]:
data.head()

,Pattern Name,Problem,Context,Solution,Result,Uses,Description,cluster
0,Reflection,Large Language Models (LLMs) often generate un...,Agentic workflows where an LLM is prompted mul...,Automate the process of critical feedback. The...,"Significant performance gains, higher-quality ...","Producing code, writing text, answering questi...",NaN,0
1,Goal Setting and Monitoring,AI agents need clear objectives to guide their...,An AI agent is tasked with achieving a specifi...,"Define explicit, measurable goals for the agen...","Focused agent behavior, ability to track progr...","Task automation, project management agents, se...",NaN,0
2,Learning and Adaptation,AI agents need to improve their performance ov...,An AI agent operates in a dynamic environment ...,Implement mechanisms for the agent to learn fr...,"Improved performance, personalization, robustn...","Personal assistants, recommendation systems, a...",NaN,0
3,Planning,AI agents need to achieve complex goals that r...,An AI agent is given a high-level goal and nee...,"The agent generates a plan (e.g., a sequence o...","Enables agents to tackle multi-step problems, ...","Task automation, robotic control, complex prob...",NaN,0
4,Prioritization,"AI agents often face multiple competing goals,...","An AI agent has limited resources (time, compu...","Implement a prioritization mechanism (e.g., ut...","More efficient resource allocation, focused ag...","Task scheduling, resource management, decision...",NaN,0


In [4]:
if not os.environ.get("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter API key for Google Gemini: ")

llm = init_chat_model("gemini-2.5-flash", model_provider="google_genai", temperature=0)

In [5]:
summarizing_prompt = """\
You are an expert AI PATTERN ANALYZER. Your task is to analyze the patterns in the dataset and provide a concise summary for each cluster.

Your output should be a JSON. Each element should contain the following fields:
- cluster_id: The unique identifier of the cluster.
- short_name: A brief, descriptive name for the cluster (max 5 words).
- description: A detailed description of the common characteristics and themes of the patterns in the cluster (make it short).
- sub_patterns: A list of all pattern names that belong to the cluster.

Here is the dataset you need to analyze:
{data}
"""

In [6]:
def save_log_summarization(cluster_id, group, summary,is_final=False):
    with open('./logs/clusters/summarization_1.txt', 'a') as f:

        f.write(f"Cluster {cluster_id}\n")
        f.write("="*100 + "\n\n")
        
        # Original Patterns (pretty JSON)
        f.write("Original Patterns:\n")
        f.write(json.dumps(group.to_dict(orient="records"), indent=4, ensure_ascii=False))
        f.write("\n\n")
        
        # Summarized Patterns (pretty JSON)
        f.write("Summarized Patterns:\n")
        f.write(json.dumps(summary, indent=4, ensure_ascii=False))
        f.write("\n\n")
        if is_final:
            f.write("="*100 + "\n")
        else:
            f.write("-"*100 + "\n")

In [9]:
import time
def summarize_cluster(cluster_df, max_retries=3, retry_delay=1.0):
    cluster_data = cluster_df.to_dict(orient="records")
    _prompt = summarizing_prompt

    last_response_text = None

    for attempt in range(1, max_retries + 1):
        prompt = _prompt.format(data=cluster_data)

        response = llm.invoke(prompt)
        last_response_text = response.content

        res_json = parse_json_safe(last_response_text, delimiter="{}")

        # ✅ check empty json
        is_empty = (
            res_json is None or
            res_json == {} or
            res_json == [] or
            (isinstance(res_json, str) and res_json.strip() == "")
        )

        if not is_empty:
            save_log_summarization(
                cluster_df["cluster"].iloc[0],
                cluster_df,
                res_json,
                is_final=True
            )
            return res_json

        # retry
        if attempt < max_retries:
            time.sleep(retry_delay)

    # ✅ final fail log
    save_log_summarization(
        cluster_df["cluster"].iloc[0],
        cluster_df,
        {"error": "Empty JSON after retries", "raw_response": last_response_text},
        is_final=True
    )

    return None

In [10]:
summarizations = []
for cluster_id, group in data.groupby('cluster'):
    print(f"Processing Cluster {cluster_id} with {len(group)} patterns...")
    summary = summarize_cluster(group)
    print(f" - {len(summary)} summarized patterns:")
    print(" - ", summary)
    summarizations.append(summary)

Processing Cluster 0 with 14 patterns...
 - 4 summarized patterns:
 -  {'cluster_id': 0, 'short_name': 'Intelligent Agent Core Capabilities', 'description': 'This cluster describes fundamental patterns for AI agents to operate autonomously and intelligently. It encompasses mechanisms for self-correction, continuous learning, adaptation to dynamic environments, structured planning, goal management, and advanced reasoning to achieve complex objectives.', 'sub_patterns': ['Reflection', 'Goal Setting and Monitoring', 'Learning and Adaptation', 'Planning', 'Prioritization', 'Exploration and Discovery', 'Dynamic Agent Knowledge Update', 'Continuous Learning Pattern', 'Reasoning Techniques', 'Reflection Pattern', 'Agentic Reasoning Loop', 'ClosedLoop Intelligence']}
Processing Cluster 1 with 11 patterns...
 - 4 summarized patterns:
 -  {'cluster_id': 1, 'short_name': 'LLM Tooling & External Actions', 'description': "This cluster encompasses patterns that extend Large Language Models' (LLMs) c

In [11]:
summarizations

[{'cluster_id': 0,
  'short_name': 'Intelligent Agent Core Capabilities',
  'description': 'This cluster describes fundamental patterns for AI agents to operate autonomously and intelligently. It encompasses mechanisms for self-correction, continuous learning, adaptation to dynamic environments, structured planning, goal management, and advanced reasoning to achieve complex objectives.',
  'sub_patterns': ['Reflection',
   'Goal Setting and Monitoring',
   'Learning and Adaptation',
   'Planning',
   'Prioritization',
   'Exploration and Discovery',
   'Dynamic Agent Knowledge Update',
   'Continuous Learning Pattern',
   'Reasoning Techniques',
   'Reflection Pattern',
   'Agentic Reasoning Loop',
   'ClosedLoop Intelligence']},
 {'cluster_id': 1,
  'short_name': 'LLM Tooling & External Actions',
  'description': "This cluster encompasses patterns that extend Large Language Models' (LLMs) capabilities beyond text generation by enabling them to interact with external systems, perform a

In [13]:
pd.DataFrame(summarizations).to_json('./clustered_dataset/cluster_summarizations-all_v2.0.json', index=False, orient='records', force_ascii=False,indent=2,lines=False)

In [20]:
summary_df = pd.DataFrame(summarizations)

In [22]:
summary_df[summary_df['sub_patterns'].apply(len) >1].to_json('./clustered_dataset/cluster_summarizations-multi_patterns-only-all_v2.0.json', index=False, orient='records', force_ascii=False,indent=2,lines=False)

In [25]:
single_pattern_summaries = summary_df[summary_df['sub_patterns'].apply(len) ==1]
single_patterns = []
for index, row in single_pattern_summaries.iterrows():
    original_pattern = data[data['Pattern Name'] == row['sub_patterns'][0]]
    print(f"Original Pattern for summary index {index}:")
    print(original_pattern.to_dict(orient='records'))
    single_patterns.append(original_pattern.to_dict(orient='records'))

import json
with open('./clustered_dataset/single_pattern_summaries-all_v2.0.json', 'w', encoding='utf-8') as f:
    json.dump(single_patterns, f, ensure_ascii=False, indent=2)


Original Pattern for summary index 10:
[{'Pattern Name': 'Feedback Loop (System-wide Improvement)', 'Problem': 'Continuously improving the overall performance, accuracy, and relevance of a complex AI system like RAG over time, based on real-world usage and outcomes.', 'Context': 'A deployed AI system (e.g., RAG) where user interactions, generated outputs, or external evaluations can provide valuable data for refinement.', 'Solution': "Implement mechanisms to iteratively feed the system's results back into its input or to finetune its trainable components. This can involve human feedback, automated evaluation metrics, or self-correction mechanisms.", 'Result': 'The system learns and adapts over time, leading to continuous improvement in performance, accuracy, and user satisfaction.', 'Uses': 'Continuous improvement of AI models and systems, adaptive learning, self-correction, human-in-the-loop systems.', 'Description': nan, 'cluster': 10}]
Original Pattern for summary index 23:
[{'Patte